# 06 — Evaluation: PatchCore vs Classical Baseline

**Author:** Sherif Abd El-Rady

**Objective:** Rigorous side-by-side evaluation of PatchCore and the classical baseline using standard anomaly detection metrics at both image and pixel level.

## Metrics

| Metric | What it measures | Why |
|---|---|---|
| **Image ROC-AUC** | Ranking quality across all thresholds | Primary metric — threshold-free |
| **Image AP** | Area under precision-recall curve | Better for class imbalance |
| **Pixel ROC-AUC** | Localisation quality | Spatial accuracy of anomaly maps |
| **IoU** | Segmentation overlap at best threshold | Practical defect segmentation quality |
| **Dice** | F1 for binary segmentation | Symmetric, handles class imbalance |


In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from defectdetect import FeatureExtractor, MemoryBank, ClassicalBaseline, AnomalyEvaluator
from defectdetect.scorer import AnomalyScorer

ROOT     = Path('..').resolve()
MVTEC    = ROOT / 'data' / 'mvtec'
CATEGORY = 'bottle'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

if not (MVTEC / CATEGORY / 'train' / 'good').exists():
    MVTEC = ROOT / 'data' / 'synthetic'; CATEGORY = ''
    print('Using synthetic dataset')

base      = MVTEC / CATEGORY if CATEGORY else MVTEC
train_dir = base / 'train' / 'good'
test_dir  = base / 'test'
gt_dir    = base / 'ground_truth'
BANK_DIR  = ROOT / 'reports' / 'memory_banks'
print('Ready')

In [ ]:
# ── 6.1  Collect test paths, labels, GT masks ──────────────────────────────────
test_paths, test_labels, gt_masks = [], [], []

for cls_dir in sorted(test_dir.iterdir()):
    if not cls_dir.is_dir(): continue
    label = 0 if cls_dir.name == 'good' else 1
    for p in sorted(cls_dir.glob('*.png')) + sorted(cls_dir.glob('*.jpg')):
        img = cv2.imread(str(p))
        if img is None: continue
        test_paths.append(p)
        test_labels.append(label)
        # Load ground truth mask if available
        if label == 1:
            mask_p = gt_dir / cls_dir.name / (p.stem + '_mask.png')
            if not mask_p.exists():
                mask_p = gt_dir / cls_dir.name / (p.stem + '.png')
            gt_masks.append(cv2.imread(str(mask_p), cv2.IMREAD_GRAYSCALE) if mask_p.exists() else None)
        else:
            gt_masks.append(None)

test_imgs  = [cv2.imread(str(p)) for p in test_paths]
test_labels = np.array(test_labels)
print(f'Test set: {len(test_paths)} images  ({test_labels.sum()} defect)')

In [ ]:
# ── 6.2  Classical baseline scores ────────────────────────────────────────────
train_imgs  = [cv2.imread(str(p)) for p in sorted(train_dir.glob('*.png'))]
baseline    = ClassicalBaseline(image_size=(224, 224), blur_sigma=2.0)
baseline.fit([img for img in train_imgs if img is not None])

bl_scores, bl_amaps = [], []
for img in tqdm(test_imgs, desc='Baseline'):
    if img is None: bl_scores.append(0.0); bl_amaps.append(None); continue
    score, amap = baseline.predict(img)
    bl_scores.append(score); bl_amaps.append(amap)

bl_scores = np.array(bl_scores)
print(f'Baseline scoring done')

In [ ]:
# ── 6.3  PatchCore scores ─────────────────────────────────────────────────────
bank_paths = sorted(BANK_DIR.glob('*.pkl'))
if not bank_paths:
    raise FileNotFoundError('Run Notebook 04 first to build the memory bank.')

bank      = MemoryBank.load(bank_paths[-1])
extractor = FeatureExtractor(layers=['layer2','layer3'], input_size=224, device=DEVICE)
scorer    = AnomalyScorer(image_size=(224, 224), gaussian_sigma=4.0)

pc_scores, pc_amaps = [], []
for img in tqdm(test_imgs, desc='PatchCore'):
    if img is None: pc_scores.append(0.0); pc_amaps.append(None); continue
    batch = extractor.preprocess([img])
    patches, sp = extractor.extract_patches(batch)
    ps = bank.score(patches, k=1)
    s, m = scorer.compute(ps, sp, n_images=1)
    pc_scores.append(float(s[0]))
    pc_amaps.append(m[0])

pc_scores = np.array(pc_scores)
pc_amaps_norm = scorer.normalise(np.stack([a for a in pc_amaps if a is not None]))
print(f'PatchCore scoring done')

In [ ]:
# ── 6.4  Image-level evaluation: ROC-AUC + AP ─────────────────────────────────
bl_eval = AnomalyEvaluator().fit(bl_scores, test_labels)
pc_eval = AnomalyEvaluator().fit(pc_scores, test_labels)

bl_metrics = bl_eval.image_level_metrics()
pc_metrics = pc_eval.image_level_metrics()

# ROC curves side by side
bl_fpr, bl_tpr, _ = bl_eval.roc_curve_data()
pc_fpr, pc_tpr, _ = pc_eval.roc_curve_data()
bl_pr, bl_rec, _  = bl_eval.pr_curve_data()
pc_pr, pc_rec, _  = pc_eval.pr_curve_data()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(bl_fpr, bl_tpr, lw=2, label=f'Classical  AUC={bl_metrics["roc_auc"]:.4f}', color='#e74c3c')
ax1.plot(pc_fpr, pc_tpr, lw=2, label=f'PatchCore  AUC={pc_metrics["roc_auc"]:.4f}', color='#3498db')
ax1.plot([0,1],[0,1],'--', color='gray', lw=1)
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC Curve — Image Level'); ax1.legend()

ax2.plot(bl_rec, bl_pr, lw=2, label=f'Classical  AP={bl_metrics["average_precision"]:.4f}', color='#e74c3c')
ax2.plot(pc_rec, pc_pr, lw=2, label=f'PatchCore  AP={pc_metrics["average_precision"]:.4f}', color='#3498db')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('PR Curve — Image Level'); ax2.legend()

plt.suptitle('PatchCore vs Classical Baseline', fontsize=12)
plt.tight_layout()
plt.show()

print(f"\n{'Metric':<25} {'Classical':>12} {'PatchCore':>12}")
print('-' * 50)
print(f"{'Image ROC-AUC':<25} {bl_metrics['roc_auc']:>12.4f} {pc_metrics['roc_auc']:>12.4f}")
print(f"{'Average Precision':<25} {bl_metrics['average_precision']:>12.4f} {pc_metrics['average_precision']:>12.4f}")
print(f"{'Best F1':<25} {bl_metrics['best_f1']:>12.4f} {pc_metrics['best_f1']:>12.4f}")

In [ ]:
# ── 6.5  Pixel-level evaluation (if GT masks available) ───────────────────────
has_masks = any(m is not None for m in gt_masks)

if has_masks:
    pc_px_scores, pc_px_labels = [], []
    bl_px_scores, bl_px_labels = [], []
    IMG_SIZE = (224, 224)

    for i, (amap_pc, amap_bl, mask) in enumerate(zip(pc_amaps_norm, bl_amaps, gt_masks)):
        if mask is None or amap_pc is None: continue
        h, w = IMG_SIZE
        m_rs  = cv2.resize(mask, (w, h))
        label_flat = (m_rs > 127).astype(int).ravel()
        pc_px_scores.extend(amap_pc.ravel().tolist())
        pc_px_labels.extend(label_flat.tolist())
        if amap_bl is not None:
            amap_bl_rs = cv2.resize(amap_bl, (w, h))
            bl_px_scores.extend(amap_bl_rs.ravel().tolist())
            bl_px_labels.extend(label_flat.tolist())

    pc_px_eval = AnomalyEvaluator().fit_pixel(np.array(pc_px_scores), np.array(pc_px_labels))
    px_metrics = pc_px_eval.pixel_level_metrics()

    print('=== Pixel-level (PatchCore) ===')
    print(f'Pixel ROC-AUC : {px_metrics["pixel_roc_auc"]:.4f}')
    print(f'Pixel AP      : {px_metrics["pixel_ap"]:.4f}')
    print(f'Best IoU      : {px_metrics["best_iou"]:.4f}')
    print(f'Dice          : {px_metrics["dice"]:.4f}')
else:
    print('No GT masks available — pixel-level evaluation skipped.')
    print('Download the full MVTec AD dataset for pixel-level metrics.')

## Results Summary

| Metric | Classical Baseline | PatchCore |
|---|---|---|
| Image ROC-AUC | *(see Cell 6.4)* | *(see Cell 6.4)* |
| Image AP | *(see Cell 6.4)* | *(see Cell 6.4)* |
| Pixel ROC-AUC | — | *(see Cell 6.5)* |
| IoU | — | *(see Cell 6.5)* |

**Published PatchCore results on MVTec AD (bottle):**  
ROC-AUC = 0.998 (WideResNet-50-2, layer2+3, ratio=0.01)

## Next step

Open `07_failure_analysis.ipynb` — examine false positives, false negatives, and hard cases.
